# Does home advantage shrink without fans? The notebook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/john-bruene/pointzerofive-data/blob/main/home-advantage-without-fans/story_003_home_advantage.ipynb)

This R notebook recomputes every statistic in
[Does home advantage shrink without fans?](https://pointzerofive.net/2026/09/home-advantage-without-fans),
story #003 on *pointzerofive*, from the raw match file. Colab opens it with the R runtime; run it
top to bottom (`Runtime → Run all`).

Every number is checked against the published results file with `stopifnot`, so the notebook
stops at the first number that does not match.

In [ ]:
library(jsonlite)

BASE <- "https://raw.githubusercontent.com/john-bruene/pointzerofive-data/main/home-advantage-without-fans/"
REPO <- ".."

# A file of this story: next to the notebook (pointzerofive-data), in the website's repository,
# else downloaded from pointzerofive-data.
fetch_path <- function(rel) {
  for (local in c(basename(rel), file.path(REPO, rel))) if (file.exists(local)) return(local)
  paste0(BASE, basename(rel))
}

near <- function(a, b, tol = 0.0015) stopifnot(abs(a - b) <= tol)

read_csv_utf8 <- function(path) {
  read.csv(path, encoding = "UTF-8", stringsAsFactors = FALSE, check.names = FALSE)
}

games <- read_csv_utf8(fetch_path("notebooks/raw/home_advantage/games_top5.csv"))
matches <- read_csv_utf8(fetch_path("notebooks/raw/bundesliga/matches.csv"))
published <- fromJSON(fetch_path("data/home-advantage-without-fans.json"), simplifyVector = FALSE)
cat(nrow(games), "league matches;", nrow(matches), "openfootball rows\n")

## 1. Two sources, one set of Bundesliga results

`games_top5.csv` comes from the transfermarkt-datasets `games` table (CC0 1.0, data from
Transfermarkt.de): every league match of the Bundesliga (`L1`), Premier League (`GB1`), LaLiga
(`ES1`), Serie A (`IT1`) and Ligue 1 (`FR1`) from 2012/13 to 2025/26. `matches.csv` is parsed
from openfootball (CC0 1.0). Club names differ between the two, so for each Bundesliga match date
we compare the list of scorelines.

In [ ]:
stopifnot(nrow(games) == published$summary$matches)
l1 <- games[games$competition_id == "L1", ]
stopifnot(all(table(l1$season) == 306))
by_date <- function(df, h, a) {
  lines <- paste(df[[h]], df[[a]], sep = "-")
  tapply(lines, df$date, function(x) paste(sort(x, method = "radix"), collapse = " "))
}
tm <- by_date(l1, "home_club_goals", "away_club_goals")
of <- by_date(matches[matches$date %in% names(tm), ], "home_goals", "away_goals")
stopifnot(setequal(names(tm), names(of)), all(tm == of[names(tm)]))
cat(sprintf("all %d Bundesliga match dates agree\n", length(tm)))

## 2. Points, home advantage and the three groups

For each match: **y = home points − away points** (win 3, draw 1, loss 0), so a home win is +3, a
draw 0 and an away win −3. Home advantage is the mean of y.

- **Ghost game:** no attendance figure between 8 March 2020 and 30 June 2022. In 2021/22 only the
  Bundesliga played without spectators again, so a missing figure in another league that season
  is a gap in the data and the match is dropped.
- **Full stadium:** attendance figure, played before 8 March 2020 or from 1 July 2022.
- **Restricted crowd:** attendance figure, played between those dates.

In [ ]:
WINDOW_START <- as.Date("2020-03-08")
WINDOW_END <- as.Date("2022-06-30")
g <- games
g$date <- as.Date(g$date)
hg <- g$home_club_goals; ag <- g$away_club_goals
g$y <- ifelse(hg > ag, 3L, ifelse(hg == ag, 1L, 0L)) - ifelse(hg < ag, 3L, ifelse(hg == ag, 1L, 0L))
in_window <- g$date >= WINDOW_START & g$date <= WINDOW_END
has_att <- !is.na(g$attendance)
g$group <- NA_character_
g$group[!in_window & has_att] <- "full"
g$group[in_window & has_att] <- "restricted"
g$group[in_window & !has_att & (g$season <= 2020 | g$competition_id == "L1")] <- "ghost"
g$cluster <- paste(g$competition_id, g$season, g$round, g$group, sep = "|")
dropped <- is.na(g$group)
kept <- g[!dropped, ]
bl <- kept[kept$competition_id == "L1", ]

print(table(g$competition_id, ifelse(dropped, "dropped", g$group)))
stopifnot(sum(dropped) == published$summary$dropped)
for (k in c("full", "restricted", "ghost")) {
  stopifnot(sum(bl$group == k) == published$summary$bundesliga[[k]])
  stopifnot(sum(kept$group == k) == published$summary$top5[[k]])
}

## 3. First results: the 2020 restart in the Bundesliga

Season 2019/20, matches before the break with full stadiums against the 83 ghost games after the
restart, counted the way headlines did.

In [ ]:
r19 <- bl[bl$season == 2019, ]
counts <- function(d) list(n = nrow(d), homeWins = sum(d$y > 0), draws = sum(d$y == 0),
                           awayWins = sum(d$y < 0), ha = mean(d$y),
                           homeGoals = mean(d$home_club_goals), awayGoals = mean(d$away_club_goals))
before <- counts(r19[r19$group == "full", ])
after <- counts(r19[r19$group == "ghost", ])
for (side in c("before", "after")) {
  mine <- get(side); pub <- published$restart[[side]]
  for (k in c("n", "homeWins", "draws", "awayWins")) stopifnot(mine[[k]] == pub[[k]])
  near(mine$ha, pub$ha); near(mine$homeGoals, pub$homeGoals, 0.006); near(mine$awayGoals, pub$awayGoals, 0.006)
}
cat(sprintf("before the break: %d-%d-%d (home wins %.0f%%), HA %+.3f\n", before$homeWins,
            before$draws, before$awayWins, 100 * before$homeWins / before$n, before$ha))
cat(sprintf("after the restart: %d-%d-%d (home wins %.0f%%), HA %+.3f\n", after$homeWins,
            after$draws, after$awayWins, 100 * after$homeWins / after$n, after$ha))

# The first matchday after the restart, 16 to 18 May 2020
md26 <- bl[bl$season == 2019 & bl$round == "26. Matchday", ]
cat(sprintf("matchday 26: %d home wins, %d draws, %d away wins\n", sum(md26$y > 0), sum(md26$y == 0), sum(md26$y < 0)))
stopifnot(sum(md26$y > 0) == 1, sum(md26$y == 0) == 3, sum(md26$y < 0) == 5)

## 4. Home advantage per Bundesliga season (Fig. 1)

In [ ]:
seasons <- do.call(rbind, lapply(2012:2025, function(s) {
  d <- bl[bl$season == s, ]
  crowd <- d[d$group != "ghost", ]; ghost <- d[d$group == "ghost", ]
  data.frame(season = sprintf("%d/%s", s, substr(s + 1, 3, 4)), n = nrow(crowd),
             ha = mean(crowd$y), nGhost = nrow(ghost),
             haGhost = if (nrow(ghost)) mean(ghost$y) else NA)
}))
print(seasons, digits = 3)
for (i in seq_len(nrow(seasons))) {
  p <- published$seasons[[i]]
  stopifnot(seasons$season[i] == p$season, seasons$n[i] == p$n, seasons$nGhost[i] == p$nGhost)
  near(seasons$ha[i], p$ha)
  if (!is.na(seasons$haGhost[i])) near(seasons$haGhost[i], p$haGhost)
}

## 5. The test: Welch's t-test

Effect = mean y in the comparison group − mean y in the ghost games, in points per match.
Positive means home advantage is smaller without spectators. The test is the two-sample t-test
without assuming equal variances in both groups (Welch); p-value and 95% interval come from the
same t distribution, so p < 0.05 exactly when the interval excludes zero.

In [ ]:
path_groups <- function(leagues, ghost, comparison) {
  d <- if (leagues == "bundesliga") bl else kept
  gh <- d[d$group == "ghost", ]
  if (ghost == "restart") gh <- gh[gh$season == 2019, ]
  if (ghost == "season2021") gh <- gh[gh$season == 2020, ]
  cmp <- switch(comparison,
    full = d[d$group == "full", ],
    before = d[d$group == "full" & d$date < WINDOW_START, ],
    any_crowd = d[d$group %in% c("full", "restricted"), ])
  list(ghost = gh, comparison = cmp)
}
welch <- function(pg) {
  t <- t.test(pg$comparison$y, pg$ghost$y, var.equal = FALSE)
  list(effect = mean(pg$comparison$y) - mean(pg$ghost$y), ci_low = t$conf.int[1],
       ci_high = t$conf.int[2], p = t$p.value, n = nrow(pg$ghost), se = unname(t$stderr),
       df = unname(t$parameter))
}

## 6. All 18 calculation paths

In [ ]:
grid <- expand.grid(comparison = c("full", "before", "any_crowd"),
                    ghost = c("all", "restart", "season2021"),
                    leagues = c("bundesliga", "top5"), stringsAsFactors = FALSE)[, 3:1]
rows <- list()
for (i in seq_len(nrow(grid))) {
  ch <- grid[i, ]
  s <- welch(path_groups(ch$leagues, ch$ghost, ch$comparison))
  rows[[i]] <- data.frame(ch, n = s$n, effect = s$effect, ci_low = s$ci_low, ci_high = s$ci_high, p = s$p)
}
specs <- do.call(rbind, rows)
print(specs, digits = 3, row.names = FALSE)
for (i in seq_len(nrow(specs))) {
  p <- published$specs[[i]]
  stopifnot(identical(unlist(p$choices), unlist(specs[i, c("leagues", "ghost", "comparison")])))
  stopifnot(specs$n[i] == p$n)
  near(specs$effect[i], p$effect, 1e-9); near(specs$ci_low[i], p$ci_low, 1e-9)
  near(specs$ci_high[i], p$ci_high, 1e-9); near(specs$p[i], p$p, 1e-9)
}
cat("all 18 paths match the published results exactly\n")

## 7. The verdict

Main calculation: Bundesliga, all ghost games, full stadiums. "Yes" needs p < 0.05 and a positive
effect. Smallest relevant effect: 0.10 points per match.

In [ ]:
main <- specs[specs$leagues == "bundesliga" & specs$ghost == "all" & specs$comparison == "full", ]
SRE <- published$claim$smallest_relevant_effect
verdict <- if (main$p < 0.05 && main$effect > 0) "yes" else "no"
kind <- if (verdict == "yes") "yes" else if (main$ci_high < SRE) "ruled out" else "not detectable"
cat(sprintf("main: %+.3f points per match, 95%% CI [%+.3f, %+.3f], p = %.3f, n = %d\n",
            main$effect, main$ci_low, main$ci_high, main$p, main$n))
cat(sprintf("verdict: %s (%s)\n", verdict, kind))
stopifnot(verdict == "no", kind == "not detectable")
yes <- specs$p < 0.05 & specs$effect > 0
cat(sprintf("yes on %d of %d paths; %d of %d top5 paths\n", sum(yes), nrow(specs),
            sum(yes & specs$leagues == "top5"), sum(specs$leagues == "top5")))

### The result table

The table under "The hypothesis test" on the website: the two group means with their t-intervals
and the difference with the Welch test.

In [ ]:
mean_row <- function(y) {
  se <- sd(y) / sqrt(length(y)); q <- qt(0.975, length(y) - 1); m <- mean(y)
  c(m, se, m - q * se, m + q * se, 2 * pt(-abs(m / se), length(y) - 1))
}
pg <- path_groups("bundesliga", "all", "full")
s <- welch(pg)
mine <- rbind(mean_row(pg$comparison$y), mean_row(pg$ghost$y), c(s$effect, s$se, s$ci_low, s$ci_high, s$p))
table_rows <- do.call(rbind, lapply(published$estimation$rows, function(r) unlist(r$values)))
stopifnot(max(abs(mine - table_rows)) < 1e-9)
dimnames(mine) <- list(c("Full stadiums", "Ghost games", "Difference (full - ghost)"),
                       c("Coefficient", "Std. Error", "Lower 95%", "Upper 95%", "Prob."))
print(round(mine, 6))

## 8. Home advantage per league and group (Fig. 2)

Mean y per league and group with its 95% t-interval.

In [ ]:
for (gp in published$groups) {
  d <- if (gp$id == "top5") kept else kept[kept$competition_id == gp$id, ]
  for (k in c("full", "restricted", "ghost")) {
    dk <- d[d$group == k, ]
    pub <- gp[[k]]
    if (is.null(pub)) { stopifnot(nrow(dk) == 0); next }
    ci <- t.test(dk$y)$conf.int
    stopifnot(nrow(dk) == pub$n)
    near(mean(dk$y), pub$ha); near(ci[1], pub$lo); near(ci[2], pub$hi)
    cat(sprintf("%-14s %-10s n=%5d  HA %+.3f [%+.3f, %+.3f]\n", gp$name, k, nrow(dk), mean(dk$y), ci[1], ci[2]))
  }
}

## Data and methods

- **Source:** transfermarkt-datasets, table `games` (CC0 1.0; version of 5 September 2026; data
  from Transfermarkt.de), <https://github.com/dcaribou/transfermarkt-datasets>. Bundesliga
  results checked against openfootball/deutschland (CC0 1.0).
- **Pipeline:** `home_advantage_pipeline.R` (base R and `jsonlite`), console output in
  `home-advantage-without-fans.output.txt`.
- **Pitch:** question, groups and forks are in `pitch.md`.
- **Licence of this notebook:** © John Brüne, see the README of pointzerofive-data.